# Algoritmo Genético Aplicado ao Problema do Caixeiro Viajante

## Especificações do Problema
O problema do caixeiro viajante consiste em descobrir a rota que torna mínima a viagem total. Nela, é necessário visitar n cidades, iniciando e encerrando sua viagem na primeira cidade visitada.


<br>

## Parâmetros da Solução
### População:
- Gerada aleatoriamente
- Formada por 8 indivíduos
- Indivíduos formados por um vetor de inteiros de 9 posições, representando a ordem em que as cidades são visitadas


<br>


### Taxa de Cruzamento
- 75%


<br>


### Taxa de Mutação
- 1%


<br>


### Critério de Parada
- Para após k iterações


<br>

## Seleção
- Será realizada pela forma de Torneio. Serão escolhidos 3 indivíduos aleatórios e selecionado o melhor entre eles (o que possui menor distância total)
- Será aplicado elitismo para garantir a presença dos mais aptos


<br>

## Função de Aptidão
- Seja $i$ um dos genes de um indivíduo da população $A$ formado por 8 valores inteiros, tal que $i = 0, 1, 2,..., 7$ e $dist(x, y)$ é a função que calcula a distância entre um par de cidades na ordem que aparecem, temos:
$$
f(A) = \sum_{i=1}^{7} dist(A(i - 1), A(i))
$$

## Cruzamento
- O cruzamento será dado por dois pontos de troca entre os cromossomos (indivíduos) definido aleatoriamente e aplicando permutação para garantir que não ocorra duplicação de cidades em uma rota.

In [1]:
# ========== BIBLIOTECAS ==========
from random import randint
from math import floor
import random
import copy
from typing import List, Tuple

In [2]:
# ========== PARÂMETROS ==========

# Função para geração da matriz de distâncias
def gera_matriz_distancias(n):
  # Cria uma matriz quadrada de tamanho n x n com zeros
  matriz = []
  for i in range(n):
      matriz.append([0]*n)

  # Coloca valores randômicos para representar as distâncias entre cidades
  for i in range(n):
        for j in range(i + 1, n):
            m = randint(5, 100)
            matriz[i][j] = m
            matriz[j][i] = m

  return matriz

# Função para transformação da rota de cidades em pares da matriz
def transforma_rota_em_pares(rota):
  pares = []
  for i in range(0, len(rota) - 1):
    # Transforma as cidades em pares
    pares.append([rota[i], rota[i + 1]])
  return pares
# Exemplo: [1, 2, 3, 4] -> [[1,2][2,3][3,4]]

# Função para transformação dos pares da matriz em rota de cidades
def transforma_pares_em_rota(pares):
  primeiro_par = pares[0]
  # Adiciona primeira cidade na rota
  rota = [primeiro_par[0]]
  # A próxima cidade da rota vai ser o segundo valor do primeiro par
  cidade_atual = primeiro_par[1]

  while len(rota) < len(pares) + 1:
      # Adiciona cidade atual à rota
      rota.append(cidade_atual)
      for a, b in pares:
          if a == cidade_atual:
              cidade_atual = b
              # Para o loop já que encontramos a próxima cidade
              break

  return rota
# Exemplo: [[1,2][2,3][3,4]] -> [1, 2, 3, 4]

# Função para calcular a distância total da rota (soma das distâncias de cada par de cidades)
def funcao_aptidao(pares, matriz):
  soma = 0
  for i, j in pares:
      soma += matriz[i - 1][j - 1]
  return soma

def ordena_populacao(populacao, matriz):
  # Ordena a população organizada em pares
  populacao.sort(key=lambda indiv: funcao_aptidao(indiv, matriz))
  return populacao

def gera_individuos(n):
  # Quantidade de genes de cada indivíduo
  k = 9
  individuos = []

  # Gera indivíduos com ordem randômica
  for i in range (0, n):
    individuo = random.sample(range(1, k-1), k-2)
    individuo.insert(0, 0)
    individuo.append(0)
    individuos.append(individuo)

  return individuos

In [3]:
# ========== OPERAÇÕES GENÉTICAS ==========
def torneio(populacao, matriz, selecao, k = 3):
    selecionados = []

    # Seleciona a quantidade de indivíduos que devem realizar cruzamento
    for i in range(selecao):

      # Seleciona k indivíduos aleatórios da população
      individuos = random.sample(populacao, k)

      individuos.sort(key=lambda indiv: funcao_aptidao(indiv, matriz))

      # Retorna o melhor indivíduo (com a menor distância)
      print("\nSelecionando melhor indivíduo do torneio...")
      print(individuos[0])
      selecionados.append(individuos[0])

    return selecionados


# Função para definir operação genética: Crossover
def crossover(populacao, taxa_crossover=0.75, n=9):

  # Transformação das rotas em pares de cidades
  populacao_rotas = []
  for p in populacao:
    populacao_rotas.append(transforma_pares_em_rota(p))

  # A nova população começa como uma cópia
  nova_populacao_rotas = populacao_rotas[:]

  # Verifica se o cruzamento deve ocorrer baseado na taxa
  if random.random() < taxa_crossover:
    print(f"\nHAVERÁ Cruzamento!")

    # Escolhe os índices de 2 indivíduos aleatórios e distintos para o cruzamento
    idx1, idx2 = random.sample(range(len(populacao_rotas)), 2)
    ind1 = populacao_rotas[idx1][:]
    ind2 = populacao_rotas[idx2][:]

    print("\nIndivíduos selecionados para cruzamento...")
    print(f"Pai 1 (índice {idx1}): {ind1}")
    print(f"Pai 2 (índice {idx2}): {ind2}")

    # Gera os pontos de cruzamento aleatoriamente
    ponto1 = random.randint(1, n - 2)
    ponto2 = random.randint(1, n - 2)

    filho1 = [None] * n
    filho1[0] = 0
    filho1[n-1] = 0

    filho2 = [None] * n
    filho2[0] = 0
    filho2[n-1] = 0

    if ponto1 > ponto2:
      ponto1, ponto2 = ponto2, ponto1

    # Realiza o crossover das regiões entre os pontos
    filho1[ponto1:ponto2] = ind1[ponto1:ponto2]
    filho2[ponto1:ponto2] = ind2[ponto1:ponto2]

    # Encontra o ponto depois da região de cruzamento
    ind2_index = ponto2
    f1_index = ponto2

    # Preenche os espaços em vazios no indivíduo gerado (filho)
    while None in filho1:
        if f1_index == n-1:
          f1_index = 1

        gene = ind2[ind2_index]

        if gene not in filho1 and gene != 0:
            filho1[f1_index] = gene
            f1_index += 1

        # Pula as posições fixas
        if ind2_index == n-1:
          ind2_index = 1
        else:
          ind2_index += 1

    # Encontra o ponto depois da região de cruzamento
    ind1_index = ponto2
    f2_index = ponto2

    while None in filho2:
        if f2_index == n-1:
          f2_index = 1

        gene = ind1[ind1_index]

        if gene not in filho2 and gene != 0:
            filho2[f2_index] = gene
            f2_index += 1

        # Pula as posições fixas
        if ind1_index == n-1:
          ind1_index = 1
        else:
          ind1_index += 1


    print(f"\nFilho 1 gerado: {filho1}")
    print(f"Filho 2 gerado: {filho2}")

    # Substitui os pais pelos filhos na nova população
    nova_populacao_rotas[idx1] = filho1
    nova_populacao_rotas[idx2] = filho2

  else:
    # Caso o cruzamento não ocorra
    print(f"\nCruzamento NÃO OCORREU! População mantida.")

  print("\nResultado final da etapa de crossover...")
  print(f"\n {nova_populacao_rotas}")

  # Retorna para a formatação de pares
  nova_populacao_final = []
  for f in nova_populacao_rotas:
    nova_populacao_final.append(transforma_rota_em_pares(f))

  return nova_populacao_final


# Função para definir operação genética: Mutação
def mutacao(populacao, prob_mutacao):

  # Transformação das rotas em pares de cidades
  populacao_rotas = []
  for p in populacao:
    populacao_rotas.append(transforma_pares_em_rota(p))

  print('\nObservação da ocorrência de mutação...')

  for individuo in populacao_rotas:
    chance = random.random() / 10

    if chance <= prob_mutacao:
      print("\nRealizando mutação...")
      print(f"\nMutação do indivíduo: {individuo}")

      ponto1 = random.randint(1, 7)
      ponto2 = random.randint(1,7)

      if ponto1 != ponto2:
        aux = individuo[ponto1]
        individuo[ponto1] = individuo[ponto2]
        individuo[ponto2] = aux

      print(f"\nMutação realizada: {individuo}")

    else:
      print(f'\nO indivíduo {individuo} não sofreu mutação!')

  # Retorna para a formatação de pares
  nova_populacao = []
  for p in populacao_rotas:
    nova_populacao.append(transforma_rota_em_pares(p))

  print("\nResultado da mutação...")
  print(f"\n {nova_populacao}")

  return nova_populacao

In [23]:
# ========== ALGORITMO EVOLUTIVO ==========
def algoritmo_evolutivo(matriz, populacao, geracoes, prob_cross, prob_mutacao, elitismo):
  print("\nTransformando cada rota em pares...")
  populacao_em_pares = [transforma_rota_em_pares(individuo) for individuo in populacao]

  print(f'\nPares gerados a partir da população:\n{populacao_em_pares}')

  # Executa cada geração até o limite
  for g in range (0, geracoes):

    # Ordena a população
    print(f"\nOrdenando a população... ({len(populacao_em_pares)} indivíduos)")
    ordena_populacao(populacao_em_pares, matriz)
    print(populacao_em_pares)

    print(f"\n ====== Melhor indivíduo da geração {g + 1} ======")
    print(populacao_em_pares[0])
    print(f"Distância total: {funcao_aptidao(populacao_em_pares[0], matriz)}")

    # Cria nova população com elitismo
    print("\nCriando nova população com elitismo...")
    populacao_mais_aptos = populacao[:elitismo]
    pares_mais_aptos = []

    for p in populacao_mais_aptos:
      pares_mais_aptos.append(transforma_rota_em_pares(p))
    print(pares_mais_aptos)

    # Realiza torneio
    individuo_torneio = torneio(populacao_em_pares, matriz, 6, k=3)

    for c in range(0, 6):
      pares_mais_aptos.append(individuo_torneio[c])

    print("\nPopulação dos mais aptos...")
    print(pares_mais_aptos)

    # Realiza crossover
    nova_populacao = crossover(pares_mais_aptos)

    # Realiza mutação
    populacao_mutacao = mutacao(nova_populacao, prob_mutacao)

    # Adicionar indivíduos na nova população, que é usada na próxima geração
    populacao_em_pares = populacao_mutacao

  populacao = [transforma_pares_em_rota(individuo) for individuo in populacao_em_pares]
  return populacao

In [25]:
# ========== MAIN ==========
n = 8
geracoes = 12
prob_cross = 0.75
prob_mutacao = 0.01
elitismo = 2

print("Gerando matriz de distâncias...")
matriz = gera_matriz_distancias(n)
print(matriz)

print("\nDefinindo a população de n indivíduos...")
populacao = gera_individuos(n)
print(populacao)

# Aplica o algoritmo evolutivo
print("\nExecutando o algoritmo evolutivo...")
populacao_final = algoritmo_evolutivo(matriz, populacao, geracoes, prob_cross, prob_mutacao, elitismo)

# Mostrar resultados
print("\nPopulação final do algoritmo evolutivo...")
print(populacao_final)

populacao_final_pares = [transforma_rota_em_pares(individuo) for individuo in populacao_final]
melhor_individuo = ordena_populacao(populacao_final_pares, matriz)[0]

print("\nO melhor indivíduo da última geração é...")
print(melhor_individuo)
print(f"Distância total: {funcao_aptidao(melhor_individuo, matriz)}")

Gerando matriz de distâncias...
[[0, 24, 27, 61, 92, 9, 97, 58], [24, 0, 85, 92, 46, 90, 18, 23], [27, 85, 0, 64, 46, 53, 89, 94], [61, 92, 64, 0, 10, 6, 100, 5], [92, 46, 46, 10, 0, 26, 40, 59], [9, 90, 53, 6, 26, 0, 94, 76], [97, 18, 89, 100, 40, 94, 0, 32], [58, 23, 94, 5, 59, 76, 32, 0]]

Definindo a população de n indivíduos...
[[0, 3, 1, 6, 5, 2, 7, 4, 0], [0, 3, 7, 1, 6, 2, 5, 4, 0], [0, 6, 2, 1, 7, 3, 5, 4, 0], [0, 4, 5, 7, 3, 1, 2, 6, 0], [0, 2, 3, 1, 7, 5, 4, 6, 0], [0, 4, 5, 6, 1, 2, 3, 7, 0], [0, 5, 3, 7, 2, 1, 4, 6, 0], [0, 5, 1, 2, 4, 7, 3, 6, 0]]

Executando o algoritmo evolutivo...

Transformando cada rota em pares...

Pares gerados a partir da população:
[[[0, 3], [3, 1], [1, 6], [6, 5], [5, 2], [2, 7], [7, 4], [4, 0]], [[0, 3], [3, 7], [7, 1], [1, 6], [6, 2], [2, 5], [5, 4], [4, 0]], [[0, 6], [6, 2], [2, 1], [1, 7], [7, 3], [3, 5], [5, 4], [4, 0]], [[0, 4], [4, 5], [5, 7], [7, 3], [3, 1], [1, 2], [2, 6], [6, 0]], [[0, 2], [2, 3], [3, 1], [1, 7], [7, 5], [5, 4], [4, 6]